# Exploratory Data Analysis
## Price Intelligence & Anomaly Detection — MrScraper

This notebook explores the training and test datasets to understand:
- Price distributions and patterns
- Temporal trends
- Discount behavior
- Price stability per product
- Anchor set coverage

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.dirname(os.getcwd()))
from src.data_loader import load_data, split_test_anchors

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
%matplotlib inline

## 1. Load Data

In [ ]:
train_df, test_df = load_data()
anchors_df, targets_df = split_test_anchors(test_df)

print(f"Training data: {train_df.shape}")
print(f"Test data: {test_df.shape}")
print(f"Anchor samples: {anchors_df.shape[0]}")
print(f"Prediction targets: {targets_df.shape[0]}")
print(f"\nDate range (train): {train_df['capturedAt'].min()} to {train_df['capturedAt'].max()}")
print(f"Date range (test): {test_df['capturedAt'].min()} to {test_df['capturedAt'].max()}")

## 2. Basic Statistics

In [ ]:
print(f"Unique shops: {train_df['shopId'].nunique()}")
print(f"Unique items: {train_df['itemId'].nunique()}")
print(f"Unique models: {train_df['modelId'].nunique()}")
print(f"Unique categories: {train_df['cat_id'].nunique()}")
print(f"Unique brands: {train_df['brand'].nunique()}")
print(f"\nPrice statistics:")
train_df['price'].describe()

## 3. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log price histogram
log_prices = np.log10(train_df['price'][train_df['price'] > 0])
axes[0].hist(log_prices, bins=80, color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].axvline(log_prices.median(), color='red', linestyle='--', label=f'Median: {10**log_prices.median():,.0f} IDR')
axes[0].set_xlabel('Log10(Price)')
axes[0].set_ylabel('Count')
axes[0].set_title('Price Distribution (Log Scale)')
axes[0].legend()

# Top categories boxplot
top_cats = train_df['cat_id'].value_counts().head(8).index
cat_data = train_df[train_df['cat_id'].isin(top_cats)].copy()
cat_data['log_price'] = np.log10(cat_data['price'].clip(lower=1))
sns.boxplot(data=cat_data, x='cat_id', y='log_price', ax=axes[1], palette='Blues_r', fliersize=1)
axes[1].set_title('Price by Top 8 Categories')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Temporal Patterns

In [ ]:
train_df['date'] = train_df['capturedAt'].dt.date
daily = train_df.groupby('date').agg(
    mean_price=('price', 'mean'),
    median_price=('price', 'median'),
    count=('price', 'count'),
    discount_rate=('priceBeforeDiscount', lambda x: (x > 0).mean() * 100)
).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(daily['date'], daily['median_price']/1e6, color='#2196F3', linewidth=2)
axes[0].set_ylabel('Median Price (Juta IDR)')
axes[0].set_title('Daily Median Price')

axes[1].bar(daily['date'], daily['count'], color='#4CAF50', alpha=0.7)
axes[1].set_ylabel('Scrape Count')
axes[1].set_title('Daily Scraping Volume')

axes[2].plot(daily['date'], daily['discount_rate'], color='#FF5722', linewidth=2)
axes[2].set_ylabel('% with Discount')
axes[2].set_title('Daily Discount Rate')

plt.tight_layout()
plt.show()

## 5. Discount Analysis

In [ ]:
has_discount = train_df['priceBeforeDiscount'] > 0
has_promotion = train_df['promotionId'] != 0

print(f"Items with discount: {has_discount.sum()} ({has_discount.mean()*100:.1f}%)")
print(f"Items with promotion: {has_promotion.sum()} ({has_promotion.mean()*100:.1f}%)")
print(f"\nDiscount percentage distribution (when > 0):")
print(train_df[train_df['show_discount'] > 0]['show_discount'].describe())

## 6. Price Stability per Product

In [ ]:
product_stats = train_df.groupby(['shopId', 'itemId', 'modelId']).agg(
    price_mean=('price', 'mean'),
    price_std=('price', 'std'),
    price_count=('price', 'count'),
).reset_index()
product_stats['price_cv'] = product_stats['price_std'] / product_stats['price_mean']
product_stats['price_cv'] = product_stats['price_cv'].fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cv_clipped = product_stats['price_cv'].clip(0, 0.5)
axes[0].hist(cv_clipped, bins=80, color='#9C27B0', alpha=0.8, edgecolor='white')
axes[0].axvline(0.05, color='red', linestyle='--', label='CV = 0.05')
axes[0].set_xlabel('Coefficient of Variation')
axes[0].set_ylabel('Number of Products')
axes[0].set_title('Price Stability (CV) per Product')
axes[0].legend()

obs_clipped = product_stats['price_count'].clip(0, 100)
axes[1].hist(obs_clipped, bins=50, color='#FF5722', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Number of Observations')
axes[1].set_ylabel('Number of Products')
axes[1].set_title('Historical Data per Product')

plt.tight_layout()
plt.show()

stable = (product_stats['price_cv'] < 0.05).mean() * 100
print(f"\n{stable:.1f}% of products have CV < 0.05 (stable prices)")
print(f"Median observations per product: {product_stats['price_count'].median():.0f}")

## 7. Anchor Set Coverage

In [ ]:
anchor_cats = anchors_df['cat_id'].nunique()
total_cats = test_df['cat_id'].nunique()
anchor_shops = anchors_df['shopId'].nunique()
total_shops = test_df['shopId'].nunique()

print(f"Anchor set coverage:")
print(f"  Categories: {anchor_cats}/{total_cats} ({anchor_cats/total_cats*100:.1f}%)")
print(f"  Shops: {anchor_shops}/{total_shops} ({anchor_shops/total_shops*100:.1f}%)")
print(f"\nAnchors per day:")
anchors_df['date'] = anchors_df['capturedAt'].dt.date
print(anchors_df.groupby('date').size())

## 8. Feature Correlations with Price

In [ ]:
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
correlations = train_df[numeric_cols].corrwith(train_df['price']).abs().sort_values(ascending=False)
print("Top correlations with price:")
print(correlations.head(15))

## 9. Key Insights

1. **Most products have stable prices** (CV < 0.05) — historical mean is the strongest predictor
2. **Discounts create bimodal distributions** — same product appears at full and discounted price
3. **Temporal trends are subtle** — gradual inflation/deflation over weeks
4. **Anchor coverage** spans most categories and shops, enabling hierarchical calibration
5. **Price distribution is highly skewed** — log transform useful for visualization, but not needed for LightGBM